# Prepare prompts for prompt engineering

Reading reduced and preprocessed traffic data from *traffic_reports_linked.csv* into `REPORTS` DataFrame. LLM should be able to generate the content of RTF file from this traffic data. Notice column **RTF_file_name** represents a link to corresponding RTF file in `/Data/promet_{year}/month_{year}` for each row DataFrame (link is just a unique RTF file name).

In [1]:
import pandas as pd

REPORTS = pd.read_csv("traffic_reports_linked.csv", encoding='utf-8')
REPORTS.head()

,Datum,A1,B1,ContentPomembnoSLO,ContentNesreceSLO,ContentZastojiSLO,ContentVremeSLO,ContentOvireSLO,ContentDeloNaCestiSLO,ContentOpozorilaSLO,ContentMednarodneInformacijeSLO,ContentSplosnoSLO,RTF_file_name
0,2022-01-01 00:07:07,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN,3871
1,2022-01-01 00:07:29,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN,3871
2,2022-01-01 00:07:30,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN,3871
3,2022-01-01 00:07:36,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN,3871
4,2022-01-01 00:16:26,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN,3871


Reading reduced RTF files from *data.json*. Column **file_name** is uased as unique identificator for linking, while **content** represents an output that LLM should be able to generate from traffic data in `REPORTS`.

In [2]:
RTFS = pd.read_csv("rtfs_reduced.csv", encoding='utf-8')
RTFS.head()

,file_name,content,datetime
0,1,Prometne informacije 23. 04. 2022 14.30 2. pro...,2022-04-23 14:30:00
1,2,Prometne informacije 23. 04. 2022 13.00 1. in ...,2022-04-23 13:00:00
2,3,Prometne informacije 08. 04. 2022 16.30 2. pro...,2022-04-08 16:30:00
3,4,Prometne informacije 17.04. 2022 07.00 1. in 2...,2022-04-17 07:00:00
4,5,Prometne informacije 17.04. 2022 06.30 1. prog...,2022-04-17 06:30:00


Splitting the data to train (70%), test (15%) and validation (15%) sets for future analysis and evaluation (only `RTFS` splitting is needed as all rows in `REPORTS` are linked to them). 

In [3]:
from sklearn.model_selection import train_test_split

RTFS_train, temp = train_test_split(RTFS, test_size=0.3, random_state=42)
RTFS_valid, RTFS_test = train_test_split(temp, test_size=0.5, random_state=42)
print(f"RTFS size: {len(RTFS)}")
print(f"RTFS_train size: {len(RTFS_train)}")
print(f"RTFS_valid size: {len(RTFS_valid)}")
print(f"RTFS_test size: {len(RTFS_test)}")

RTFS size: 28037
RTFS_train size: 19625
RTFS_valid size: 4206
RTFS_test size: 4206


Functions for generating few-shot LLM prompt.

In [5]:
def prepare_input(RTF_file_name):

    # Traffic data for the given RTF file name
    reports = REPORTS[REPORTS['RTF_file_name'] == RTF_file_name]

    # Input data used for generating desired LLM output
    input = {col: set(reports[col].dropna()) for col in [
        'Datum',
        'A1', 
        'B1', 
        'ContentPomembnoSLO', 
        'ContentNesreceSLO', 
        'ContentZastojiSLO', 
        'ContentVremeSLO', 
        'ContentOvireSLO', 
        'ContentDeloNaCestiSLO', 
        'ContentOpozorilaSLO',
        'ContentMednarodneInformacijeSLO', 
        'ContentSplosnoSLO']}
    
    lines = ["Vhodni podatki:"]
    
    if input['ContentPomembnoSLO']:
        lines.append(f"- Zelo pomembne informacije o prometu: {', '.join(map(str, input['ContentPomembnoSLO']))}")
    if input['A1']:
        lines.append(f"- Pomembne informacije o prometu: {', '.join(map(str, input['A1']))}")
    if input['B1']:
        lines.append(f"- Manj pomembne informacije o prometu: {', '.join(map(str, input['B1']))}")
    if input['ContentNesreceSLO']:
        lines.append(f"- Informacije o nesrečah: {', '.join(map(str, input['ContentNesreceSLO']))}")
    if input['ContentZastojiSLO']:
        lines.append(f"- Informacije o zastojih: {', '.join(map(str, input['ContentZastojiSLO']))}")
    if input['ContentVremeSLO']:
        lines.append(f"- Informacije o vremenu: {', '.join(map(str, input['ContentVremeSLO']))}")
    if input['ContentOvireSLO']:
        lines.append(f"- Informacije o ovirah: {', '.join(map(str, input['ContentOvireSLO']))}")
    if input['ContentDeloNaCestiSLO']:
        lines.append(f"- Informacije o delu na cesti: {', '.join(map(str, input['ContentDeloNaCestiSLO']))}")
    if input['ContentOpozorilaSLO']:
        lines.append(f"- Informacije o opozorilih: {', '.join(map(str, input['ContentOpozorilaSLO']))}")
    if input['ContentMednarodneInformacijeSLO']:
        lines.append(f"- Informacije o mednarodnih informacijah: {', '.join(map(str, input['ContentMednarodneInformacijeSLO']))}")
    if input['ContentSplosnoSLO']:
        lines.append(f"- Splošne informacije: {', '.join(map(str, input['ContentSplosnoSLO']))}")

    return lines

def generate_shot(RTFS_, RTF_file_name):
    
    # Preparing example shot text
    shot = prepare_input(RTF_file_name)

    # Desired LLM output
    output = RTFS_[RTFS_['file_name'] == RTF_file_name]["content"].values[0]

    shot.append("")
    shot.append(f"Poročilo: {output}\n")

    return '\n'.join(shot)

# Example usage
print(generate_shot(RTFS_train, 1))


Vhodni podatki:

Poročilo: Prometne informacije 23. 04. 2022 14.30 2. program Podatki o prometu. Na cesti Stara Vrhnika-Podlipa je zaradi nesreče v Podlipi promet urejen izmenično enosmerno. Na ljubljanski vzhodni obvoznici proti Mariboru je zaradi živali na vozišču oviran promet pred pokritim vkopom Strmec. Na mejnem prehodu Obrežje vozniki osebnih vozil na vstop v državo čakajo do 2 uri, v Gruškovju pa približno 1 uro. 


Combining all data to prepare an example of few-shot prompt for test sample `random_test_RTF`. As we can see in shot examples in cell's output, informations from traffic data `REPORTS` and informations from corresponding `RTF` content **DO NOT MATCH**. We then manually checked some rows in `.xlsx` file and their `.rtf` file based on same the date and conclude that the problem is not in our linking but in the given data. **We assume there were additional sources used in writing `.rtf` files.**

In [7]:
# Randomly select a test RTF file name and prepare the input data
random_test_RTF = RTFS_test.sample(1).iloc[0]['file_name']

# Ipnut data lines for the test sample
test_data = '\n'.join(prepare_input(random_test_RTF))

# Generate the few-shot prompt with some training examples
few_shot_prompt = f"""Generiraj prometno poročilo na podlagi spodnjih vhodnih podatkov:
{test_data}

Spodaj je podanih par primerov vhodnih poročil in željenih izhodnih poročil iz teh podatkov:

{generate_shot(RTFS_train, 1)}
{generate_shot(RTFS_train, 2)}
{generate_shot(RTFS_train, 3)}
"""
# Example of few-shot prompt
print(few_shot_prompt)
with open("few_shot_prompt.txt", "w", encoding="utf-8") as f:
    f.write(few_shot_prompt)

Generiraj prometno poročilo na podlagi spodnjih vhodnih podatkov:
Vhodni podatki:
- Manj pomembne informacije o prometu: Na vipavski hitri cesti med počivališčem Šempas in Selom proti Nanosu zaradi okvare vozila zaprt vozni pas.Proti Vrhniki, Kopru:- Na ljubljanski južni obvoznici od Malenc mimo Viča.- Na gorenjski in zahodni obvoznici od Šmartnega mimo Kosez in Brda. Občasno zapirajo predor Šentvid.Na mariborski vzhodni obvoznici pri Malečniku v obe smeri. Zamuda 10 - 15 minut.Na cestah Lesce - Bled, Vič - Brezovica, Lucija - Strunjan, Izola - Strunjan in Dragonja - Šmarje - Koper v obe smeri.Zaprta je cesta Nova vas - Pišece, pri Brezovici predvidoma do 28. julija.Več o delovnih zaporah v prometni napovedi., Proti Vrhniki, Kopru:- Na ljubljanski južni obvoznici od Malenc mimo Viča.- Na gorenjski in zahodni obvoznici od Šmartnega mimo Kosez in Brda. Občasno zapirajo predor Šentvid.Na mariborski vzhodni obvoznici pri Malečniku v obe smeri. Zamuda 10 - 15 minut.Na cestah Lesce - Bled, V